# MCTS evaluation on Kaggle (CPU)

Paired evaluation of the Monte Carlo tree search of [CL-BZH/Pacman](https://github.com/CL-BZH/Pacman) on game v2:
the agent alone, the MCTS with the exact ghost model and the MCTS with the estimated (fair) ghost model,
on the same games (`Tools/mcts_eval.py`).

**Before running:** in the notebook settings (right panel) set *Accelerator → None* (CPU only: this does not use
the weekly GPU quota) and turn *Internet* on (needed to clone the repository).

The search runs one simulation at a time and is CPU-bound: about 5 minutes per game per core, so 150 games of
each search take about 6–7 hours on Kaggle's 4 cores. Use *Save Version → Save & Run All (Commit)* so that the
run continues with the browser closed. Each batch of 10 games is written to `mcts_eval.jsonl` as soon as it
ends; the two searches are played batch by batch in turn, so a run stopped early still gives paired results.

## 1. Setup

In [ ]:
import os
REPO_DIR = "/kaggle/working/Pacman"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/CL-BZH/Pacman.git {REPO_DIR}
assert os.path.isdir(REPO_DIR), "Cloning the repository failed: check that Internet is ON in the notebook settings."
%cd {REPO_DIR}
print("CPU cores:", os.cpu_count())

## 2. Configuration

In [ ]:
CHECKPOINT = "Trainings/A2C/PPO_v2/pacman_a2c_100M.pth"    # the agent: run 6 (PPO only, 100M steps);
                                                          # pacman_a2c_final.pth (expert iteration) learned
                                                          # from the MCTS with the exact ghost model
GAMES      = 150        # games per configuration (a multiple of 10)
SEED       = 4001       # seed of the first game
SIMS       = 100        # simulations per search
HORIZON    = 10         # moves of the agent after a new node before its value
BATCH      = 1          # simulations per wave: 1 = one at a time (waves weaken the search)
CONFIGS    = "agent,exact,estimated"
OUT        = "/kaggle/working/mcts_eval.jsonl"
# To continue a stopped run: add the previous version's output as an input (Add Input) and copy it, e.g.
# !cp /kaggle/input/<name>/mcts_eval.jsonl {OUT}     (the batches already played are not replayed)

## 3. Run

In [ ]:
!python3 Tools/mcts_eval.py {CHECKPOINT} --games {GAMES} --seed {SEED} --sims {SIMS} --horizon {HORIZON} \
    --batch {BATCH} --configs {CONFIGS} --out {OUT}

## 4. Summary

Win rates on the games played by all configurations, and the sign tests of the paired comparisons.

In [ ]:
!python3 Tools/mcts_eval.py {CHECKPOINT} --summary --out {OUT}